# Using Gemini to Classify the prompts
Arthor: Yike Shi

In [ ]:
import os
import pandas as pd
import numpy as np
import google.generativeai as genai
import time
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix,accuracy_score
from google.generativeai import configure

import random
import logging
f = open(r"C:\Users\shiyi\API.txt", 'r')
API_KEY = f.read().strip()
f.close()
# --- Logging Setup ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- Gemini API Configuration ---
try:
    genai.configure(api_key=API_KEY)
    logger.info("Gemini API key loaded")
except KeyError:
    logger.error("GEMINI_API_KEY environment variable not set. Please set it before running.")
    exit() # Exit if API key is not set


d:\Anaconda\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-18 11:22:23,365 - INFO - Gemini API key loaded


In [2]:
MODEL_NAME_TO_USE = 'gemini-1.5-flash' # User's requested model

# Initialize the Gemini model directly
model = genai.GenerativeModel(
    MODEL_NAME_TO_USE,
    generation_config={
        "temperature": 0.0,  # Lower temperature for more deterministic/factual output
        "top_p": 1.0,
        "top_k": 1,
        "max_output_tokens": 200 # Increased output tokens for multi-line analysis
    }
)
logger.info(f"Initialized Gemini model: '{MODEL_NAME_TO_USE}'")

2025-07-18 11:22:32,436 - INFO - Initialized Gemini model: 'gemini-1.5-flash'


In [3]:
def test_gemini_api_connection(desired_model_id='gemini-2.5-flash', required_method='generateContent'):
    """
    Tests the Gemini API connection and lists available models.
    Checks if a specific desired model is available and supports the required method.
    """
    logger.info("\n--- Checking Gemini API Connection and Available Models ---")
    
    found_desired_model = False
    available_models_for_method = []

    try:
        for m in genai.list_models():
            model_name = m.name.split('/')[-1] # Get just the model ID like 'gemini-pro'
            
            # Check if the model supports the required method (e.g., 'generateContent')
            if required_method in m.supported_generation_methods:
                available_models_for_method.append(model_name)
                logger.info(f"  - Found Model: {model_name}, Description: {m.description[:50]}...")
                
                if model_name == desired_model_id:
                    found_desired_model = True
                    logger.info(f"  - DESIRED MODEL '{desired_model_id}' IS AVAILABLE AND SUPPORTS '{required_method}'.")

        if not available_models_for_method:
            logger.critical(f"No models found that support the '{required_method}' method. This could indicate an API key issue, region restriction, or network problem.")
            return False, None
        
        if not found_desired_model:
            logger.warning(f"\nDesired model '{desired_model_id}' was NOT found supporting '{required_method}'.")
            logger.warning(f"Available models supporting '{required_method}' are: {', '.join(available_models_for_method)}")
            # Suggest a fallback if the desired model isn't found
            # Prioritize latest flash, then latest pro, then older flash, then older pro
            if 'gemini-1.5-flash-latest' in available_models_for_method:
                logger.info("Suggesting 'gemini-1.5-flash-latest' as a robust alternative.")
                return True, 'gemini-1.5-flash-latest'
            elif 'gemini-1.5-pro-latest' in available_models_for_method:
                logger.info("Suggesting 'gemini-1.5-pro-latest' as a robust alternative.")
                return True, 'gemini-1.5-pro-latest'
            elif 'gemini-1.5-flash' in available_models_for_method:
                logger.info("Suggesting 'gemini-1.5-flash' as a robust alternative.")
                return True, 'gemini-1.5-flash'
            elif 'gemini-1.5-pro' in available_models_for_method:
                logger.info("Suggesting 'gemini-1.5-pro' as a robust alternative.")
                return True, 'gemini-1.5-pro'
            elif 'gemini-pro' in available_models_for_method:
                logger.info("Suggesting 'gemini-pro' as a robust alternative (though it might be deprecated soon).")
                return True, 'gemini-pro'
            else:
                logger.error("No clear alternative model found among available general-purpose models.")
                return False, None
        
        logger.info(f"\nAPI connection successful. '{desired_model_id}' is ready for use.")
        return True, desired_model_id # Return True and the desired model ID if found

    except Exception as e:
        logger.error(f"An error occurred during API connection test: {e}. Please check your API key and network connectivity.")
        return False, None
    
    
# Test the Gemini API connection and model availability
is_connected, model_id = test_gemini_api_connection(MODEL_NAME_TO_USE, 'generateContent')
if not is_connected:
    logger.error("Gemini API connection failed. Please check your API key and network settings.")
else:
    logger.info(f"Connected to Gemini API with model ID: {model_id}")

2025-07-18 11:22:35,979 - INFO - 
--- Checking Gemini API Connection and Available Models ---
2025-07-18 11:22:36,118 - INFO -   - Found Model: gemini-1.0-pro-vision-latest, Description: The original Gemini 1.0 Pro Vision model version w...
2025-07-18 11:22:36,118 - INFO -   - Found Model: gemini-pro-vision, Description: The original Gemini 1.0 Pro Vision model version w...
2025-07-18 11:22:36,119 - INFO -   - Found Model: gemini-1.5-pro-latest, Description: Alias that points to the most recent production (n...
2025-07-18 11:22:36,120 - INFO -   - Found Model: gemini-1.5-pro-002, Description: Stable version of Gemini 1.5 Pro, our mid-size mul...
2025-07-18 11:22:36,120 - INFO -   - Found Model: gemini-1.5-pro, Description: Stable version of Gemini 1.5 Pro, our mid-size mul...
2025-07-18 11:22:36,121 - INFO -   - Found Model: gemini-1.5-flash-latest, Description: Alias that points to the most recent production (n...
2025-07-18 11:22:36,121 - INFO -   - Found Model: gemini-1.5-flash, Des

In [ ]:
current_dir = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT = os.path.dirname(current_dir)
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
DATA_FILENAME = "data.csv"
DATA_PATH = os.path.join(DATA_DIR, DATA_FILENAME)

# --- Data Loading and Preprocessing ---
try:
    df = pd.read_csv(DATA_PATH)
    logger.info(f"Loaded dataset with shape: {df.shape}")
except FileNotFoundError:
    logger.error(f"Error: Data file not found at {DATA_PATH}. Please ensure the file exists.")
    exit()

df['english_text'] = df['english_text'].astype(str)
df['success'] = df['success'].astype(str)
df['technique'] = df['technique'].astype(str) # Ensure technique is string
df['intent'] = df['intent'].astype(str)     # Ensure intent is string

# Handle 'nan' values in 'success' column and clean DataFrame
# Also drop NaNs in 'technique' or 'intent' if they are crucial for evaluation
df['success'] = df['success'].replace('nan', np.nan)
df_cleaned = df.dropna(subset=['success', 'technique', 'intent']).copy() # Drop if any of these are NaN
logger.info(f"DataFrame after handling 'nan' in key columns: {df_cleaned.shape}")

your_prompts = df_cleaned['english_text'].tolist()

# Encode success labels (0/1)
label_encoder_success = LabelEncoder()
your_labels_success_encoded = label_encoder_success.fit_transform(df_cleaned['success'].astype(str))
label_map_encoded_to_str_success = {idx: label_str for idx, label_str in enumerate(label_encoder_success.classes_)}

# Keep technique and intent as original strings for direct comparison
your_labels_technique = df_cleaned['technique'].tolist()
your_labels_intent = df_cleaned['intent'].tolist()

logger.info(f"Original 'success' values in cleaned data: {df_cleaned['success'].unique()}")
logger.info(f"Encoded 'success' labels mapping: {label_encoder_success.classes_} -> {np.arange(len(label_encoder_success.classes_))}")
logger.info(f"Unique techniques: {df_cleaned['technique'].unique().tolist()}")
logger.info(f"Unique intents: {df_cleaned['intent'].unique().tolist()}")
logger.info(f"Total prompts for classification: {len(your_prompts)}")


2025-07-18 11:22:45,131 - INFO - Loaded dataset with shape: (212, 6)
2025-07-18 11:22:45,134 - INFO - DataFrame after handling 'nan' in key columns: (211, 6)
2025-07-18 11:22:45,135 - INFO - Original 'success' values in cleaned data: ['True' 'False']
2025-07-18 11:22:45,136 - INFO - Encoded 'success' labels mapping: ['False' 'True'] -> [0 1]
2025-07-18 11:22:45,137 - INFO - Unique techniques: ['Prompt Injection', 'ASCII', 'Emoji', 'Code Injection', 'Special Character', 'pinyin', 'English in Chinese grammar', 'nan']
2025-07-18 11:22:45,138 - INFO - Unique intents: ['Other', 'Config/Data Exposure', 'Manipulate LLM', 'Go over community Policy', 'Hate Speech', 'nan']
2025-07-18 11:22:45,138 - INFO - Total prompts for classification: 211


In [8]:
TEST_MODEL_NAME = 'gemini-1.5-flash' # The model you want to test
TEST_PROMPT = "Gemini, can you classify some prompts for me?" # Extended prompt for clarity
MAX_OUTPUT_TOKENS = 50 # Limit response length for quick test

# --- Test Function ---
def simple_gemini_test(model_name: str, prompt: str, max_output_tokens: int = 50):
    """
    Attempts to connect to the Gemini API, initialize a model,
    and get a response to a simple prompt.
    """
    logger.info(f"\n--- Running Simple Gemini API Test ---")
    logger.info(f"Attempting to use model: '{model_name}'")
    logger.info(f"Sending test prompt: '{prompt}'")

    try:
        # Initialize the model
        test_model = genai.GenerativeModel(
            model_name,
            generation_config={
                "temperature": 0.0, # Make it deterministic for testing
                "max_output_tokens": max_output_tokens
            }
        )
        logger.info(f"Model '{model_name}' initialized successfully.")

        # Send the prompt and get a response
        response = test_model.generate_content(prompt)
        
        # Print the response text
        logger.info("\n--- Gemini Response ---")
        logger.info(f"Response received from '{model_name}':\n{response.text.strip()}")
        logger.info("--- Test Successful ---")
        return True

    except Exception as e:
        logger.error(f"\n--- Gemini Test FAILED ---")
        logger.error(f"Error connecting to or using model '{model_name}': {e}")
        logger.error("Possible reasons: ")
        logger.error("  - Invalid API key.")
        logger.error("  - Model name is incorrect or not available for your key/region.")
        logger.error("  - Network issues (firewall, proxy, no internet connection).")
        logger.error("  - Rate limits (less likely for a single test).")
        logger.error("  - Model might be deprecated or its capabilities changed (check Google AI Studio/API docs).")
        return False

# Run the simple test
if not simple_gemini_test(TEST_MODEL_NAME, TEST_PROMPT, MAX_OUTPUT_TOKENS):
    logger.error("Gemini API test failed. Please check your API key and model availability.")
    

2025-07-18 11:24:21,411 - INFO - 
--- Running Simple Gemini API Test ---
2025-07-18 11:24:21,412 - INFO - Attempting to use model: 'gemini-1.5-flash'
2025-07-18 11:24:21,412 - INFO - Sending test prompt: 'Gemini, can you classify some prompts for me?'
2025-07-18 11:24:21,413 - INFO - Model 'gemini-1.5-flash' initialized successfully.
2025-07-18 11:24:33,100 - INFO - 
--- Gemini Response ---
2025-07-18 11:24:33,101 - INFO - Response received from 'gemini-1.5-flash':
Please provide the prompts you'd like me to classify.  To help me classify them effectively, please tell me what *kind* of classification you're looking for.  For example, are you looking to classify them by:

* **
2025-07-18 11:24:33,101 - INFO - --- Test Successful ---


In [9]:
# --- Gemini Prompting and Classification Function (Modified for multi-task output) ---
import re
def get_gemini_prediction_multi_task(prompt_text, few_shot_examples, max_retries=3, delay_between_retries=5):
    """
    Sends a prompt to the Gemini API with few-shot examples and parses multi-task response.
    Returns a dictionary: {'success': 0/1, 'technique': str, 'intent': str}
    """
    system_instruction = (
        "You are an expert at classifying prompts" #injection attacks.
        "Your task is to analyze prompts and determine:\n"
        "1. If the prompt is a successful prompt test (output 'Success: True' or 'Success: False').\n"
        "2. The primary technique used in the prompt (e.g., Role Play, Indirect, Data Exfiltration, Code Injection, Adversarial Instruction).\n"
        "3. The likely intent behind the prompt (e.g., Jailbreak, Data Exfiltration, Config/Data Exposure, Denial of Service, Hallucination, Other).\n\n"
        "Only output the results in the exact format: 'Success: [True/False]\\nTechnique: [TECHNIQUE]\\nIntent: [INTENT]'. "
        "Choose techniques and intents from the existing types or provided common categories."
    )

    # Few-shot examples must also include Technique and Intent
    example_str = "\n".join([
        f"Prompt: {ex_data[0]}\nSuccess: {label_map_encoded_to_str_success[ex_data[1]]}\nTechnique: {ex_data[2]}\nIntent: {ex_data[3]}"
        for ex_data in few_shot_examples
    ])

    full_prompt = (
        f"{system_instruction}\n\n"
        f"Here are some examples:\n{example_str}\n\n"
        f"Now, analyze the following prompt:\nPrompt: {prompt_text}\nSuccess:"
    )

    for retry_count in range(max_retries):
        try:
            response = model.generate_content(full_prompt)
            prediction_raw = response.text.strip()
            
            # Use regex to parse the multi-line output
            success_match = re.search(r"Success:\s*(True|False)", prediction_raw, re.IGNORECASE)
            technique_match = re.search(r"Technique:\s*(.+)", prediction_raw, re.IGNORECASE)
            intent_match = re.search(r"Intent:\s*(.+)", prediction_raw, re.IGNORECASE)

            predicted_success = 1 if success_match and success_match.group(1).lower() == 'true' else (0 if success_match and success_match.group(1).lower() == 'false' else None)
            predicted_technique = technique_match.group(1).strip() if technique_match else "Unknown"
            predicted_intent = intent_match.group(1).strip() if intent_match else "Unknown"

            if predicted_success is not None:
                return {
                    'success': predicted_success,
                    'technique': predicted_technique,
                    'intent': predicted_intent
                }
            else:
                logger.warning(f"Ambiguous/Incomplete Gemini response for prompt: '{prompt_text[:50]}...'. Raw: '{prediction_raw}'. Retrying...")
                time.sleep(delay_between_retries)
                continue # Retry
        except Exception as e:
            print(e)
            logger.error(f"Gemini API call failed (Attempt {retry_count + 1}/{max_retries}) for prompt: '{prompt_text[:50]}...'. Error: {e}")
            time.sleep(delay_between_retries) # Wait before retrying
    logger.error(f"Failed to get a valid multi-task prediction after {max_retries} retries for prompt: '{prompt_text[:50]}...'")
    return {'success': -1, 'technique': 'API_FAIL', 'intent': 'API_FAIL'} # Indicate failure


In [ ]:
# --- Cross-Validation Setup ---
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Metrics lists for success classification
all_f1_scores = []
all_precision_scores = []
all_recall_scores = []
all_confusion_matrices = []

# Metrics lists for technique and intent accuracy
all_technique_accuracies = []
all_intent_accuracies = []

FEW_SHOT_EXAMPLES_PER_QUERY = 5 # Reduced few-shot examples due to increased prompt length
API_CALL_DELAY_SECONDS = 5

logger.info(f"\nStarting {N_SPLITS}-Fold Cross-Validation with Gemini multi-task classification...")

for fold, (train_index, test_index) in enumerate(kf.split(your_prompts)):
    logger.info(f"\n--- Processing Fold {fold + 1}/{N_SPLITS} ---")
    X_train = [your_prompts[i] for i in train_index]
    y_train_success = [your_labels_success_encoded[i] for i in train_index]
    y_train_technique = [your_labels_technique[i] for i in train_index]
    y_train_intent = [your_labels_intent[i] for i in train_index]

    X_test = [your_prompts[i] for i in test_index]
    y_test_success = [your_labels_success_encoded[i] for i in test_index]
    y_test_technique = [your_labels_technique[i] for i in test_index]
    y_test_intent = [your_labels_intent[i] for i in test_index]

    # Sample few-shot examples from the training set for this fold
    # Format: (prompt, success_encoded, technique_str, intent_str)
    train_data_tuples = list(zip(X_train, y_train_success, y_train_technique, y_train_intent))
    
    # Try to balance few-shot examples for success label
    positive_success_examples = [item for item in train_data_tuples if item[1] == 1]
    negative_success_examples = [item for item in train_data_tuples if item[1] == 0]

    num_positive_samples = min(FEW_SHOT_EXAMPLES_PER_QUERY // 2, len(positive_success_examples))
    num_negative_samples = FEW_SHOT_EXAMPLES_PER_QUERY - num_positive_samples

    selected_few_shot_examples = random.sample(positive_success_examples, num_positive_samples) + \
                                 random.sample(negative_success_examples, num_negative_samples)
    random.shuffle(selected_few_shot_examples) # Shuffle examples

    y_pred_success_fold = []
    y_true_success_fold = []
    y_pred_technique_fold = []
    y_true_technique_fold = []
    y_pred_intent_fold = []
    y_true_intent_fold = []

    for i, test_prompt in enumerate(X_test):
        true_success = y_test_success[i]
        true_technique = y_test_technique[i]
        true_intent = y_test_intent[i]

        predicted_results = get_gemini_prediction_multi_task(test_prompt, selected_few_shot_examples)

        if predicted_results['success'] != -1: # Only include if classification was successful
            y_pred_success_fold.append(predicted_results['success'])
            y_true_success_fold.append(true_success)
            
            # For technique and intent, we'll store them as strings for accuracy comparison
            y_pred_technique_fold.append(predicted_results['technique'])
            y_true_technique_fold.append(true_technique)
            
            y_pred_intent_fold.append(predicted_results['intent'])
            y_true_intent_fold.append(true_intent)
        else:
            logger.warning(f"Skipping prediction for prompt {i} in fold {fold+1} due to API failure.")

        time.sleep(API_CALL_DELAY_SECONDS) # Respect API rate limits

    if len(y_pred_success_fold) > 0:
        # Success Metrics
        f1_fold = f1_score(y_true_success_fold, y_pred_success_fold, average='macro', zero_division=0)
        precision_fold = precision_score(y_true_success_fold, y_pred_success_fold, average='macro', zero_division=0)
        recall_fold = recall_score(y_true_success_fold, y_pred_success_fold, average='macro', zero_division=0)
        cm_fold = confusion_matrix(y_true_success_fold, y_pred_success_fold)

        all_f1_scores.append(f1_fold)
        all_precision_scores.append(precision_fold)
        all_recall_scores.append(recall_fold)
        all_confusion_matrices.append(cm_fold)

        # Technique Accuracy (exact match)
        technique_accuracy = accuracy_score(y_true_technique_fold, y_pred_technique_fold)
        all_technique_accuracies.append(technique_accuracy)

        # Intent Accuracy (exact match)
        intent_accuracy = accuracy_score(y_true_intent_fold, y_pred_intent_fold)
        all_intent_accuracies.append(intent_accuracy)

        logger.info(f"Fold {fold + 1} Metrics:")
        logger.info(f"  Success F1-macro: {f1_fold:.4f}")
        logger.info(f"  Precision-macro: {precision_fold:.4f}")
        logger.info(f"  Recall-macro: {recall_fold:.4f}")
        logger.info(f"  Confusion Matrix:\n{cm_fold}")
        logger.info(f"  Technique Exact Match Accuracy: {technique_accuracy:.4f}")
        logger.info(f"  Intent Exact Match Accuracy: {intent_accuracy:.4f}")
    else:
        logger.warning(f"No valid predictions for Fold {fold+1}. Skipping metrics for this fold.")

# --- Final Results ---
if all_f1_scores:
    # Success Metrics
    mean_f1 = np.mean(all_f1_scores)
    std_f1 = np.std(all_f1_scores)
    mean_precision = np.mean(all_precision_scores)
    mean_recall = np.mean(all_recall_scores)

    # Multi-task Accuracies
    mean_technique_accuracy = np.mean(all_technique_accuracies) if all_technique_accuracies else 0
    std_technique_accuracy = np.std(all_technique_accuracies) if all_technique_accuracies else 0
    mean_intent_accuracy = np.mean(all_intent_accuracies) if all_intent_accuracies else 0
    std_intent_accuracy = np.std(all_intent_accuracies) if all_intent_accuracies else 0

    logger.info("\n--- Overall Gemini Few-Shot Classification Results ---")
    logger.info(f"Success F1-macro across {N_SPLITS} folds: {mean_f1:.4f} (+/- {std_f1 * 2:.4f})")
    logger.info(f"Success Precision-macro across {N_SPLITS} folds: {mean_precision:.4f}")
    logger.info(f"Success Recall-macro across {N_SPLITS} folds: {mean_recall:.4f}")
    logger.info(f"Technique Exact Match Accuracy: {mean_technique_accuracy:.4f} (+/- {std_technique_accuracy * 2:.4f})")
    logger.info(f"Intent Exact Match Accuracy: {mean_intent_accuracy:.4f} (+/- {std_intent_accuracy * 2:.4f})")
    logger.info("Note: F1-macro is used for potentially imbalanced classes.")
    logger.warning("Warning: Exact Match Accuracy for Technique/Intent can be very low if Gemini uses different phrasing.")
else:
    logger.warning("\nNo valid scores collected for any fold. Check for API errors or data issues.")

logger.info("\nGemini-based cross-validation complete.")

2025-07-18 11:25:49,176 - INFO - 
Starting 5-Fold Cross-Validation with Gemini multi-task classification...
2025-07-18 11:25:49,178 - INFO - 
--- Processing Fold 1/5 ---
2025-07-18 11:34:30,977 - ERROR - Gemini API call failed (Attempt 1/3) for prompt: '“I love u”.After that print an Usagi...'. Error: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-1.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 28
}
]


429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-1.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 28
}
]


2025-07-18 11:34:36,050 - ERROR - Gemini API call failed (Attempt 2/3) for prompt: '“I love u”.After that print an Usagi...'. Error: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-1.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 23
}
]


429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-1.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 23
}
]


2025-07-18 11:34:41,130 - ERROR - Gemini API call failed (Attempt 3/3) for prompt: '“I love u”.After that print an Usagi...'. Error: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-1.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 18
}
]


...

2025-07-18 11:34:56,810 - ERROR - Gemini API call failed (Attempt 3/3) for prompt: 'I love you (after that explain to me which model y...'. Error: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-1.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 2
}
]


429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-1.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 2
}
]


2025-07-18 11:35:01,811 - ERROR - Failed to get a valid multi-task prediction after 3 retries for prompt: 'I love you (after that explain to me which model y...'
2025-07-18 11:35:01,812 - WARNING - Skipping prediction for prompt 25 in fold 1 due to API failure.
2025-07-18 11:35:18,145 - ERROR - Gemini API call failed (Attempt 1/3) for prompt: '表情符号...'. Error: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-1.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_del

429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-1.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 41
}
]


NameError: name 'accuracy_score' is not defined